# Halocin C8 data prep 2: GTDB only hits

Subset of halocin C8 hits present in GTDB (i.e. excluding UniProtKB only hits)

In [1]:
import copy
import gzip
import os
from pathlib import Path
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO, Entrez

cwd = os.getcwd()
if cwd.endswith('Halocins'):
    os.chdir('../..')
    cwd = os.getcwd()

In [2]:
sns.set_palette('colorblind')
sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.8)
plt.rcParams['font.family'] = 'Helvetica'

palette = sns.color_palette().as_hex()

data_folder = Path('data/')
assert data_folder.is_dir()

amp_db_folder = data_folder / 'amp_db'
assert amp_db_folder.is_dir()

halocins_folder = amp_db_folder / 'Halocins'
assert halocins_folder.is_dir()

halc8_folder = halocins_folder / 'Halocin_C8'
assert halc8_folder.is_dir()

halc8_out = Path('data/outputs/amp_db/Halocins/Halocin_C8')


In [85]:
halc8_hits = pd.read_csv(halc8_out / 'sequences' / 'HalC8_proteins.csv', index_col='id')
print(f'Halocin C8 hits: {len(halc8_hits)}')

halc8_hits = halc8_hits[halc8_hits['gtdb_phylum'].notnull()].copy()
print(f'Halocin C8 hits in GTDB: {len(halc8_hits)}')

halc8_hits.head()

Halocin C8 hits: 253
Halocin C8 hits in GTDB: 137


,uniprot_id,gtdb_id,db_proka_id,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,length,source
id,,,,,,,,,,,,,
O28702,O28702,NC_000917.1_1621,NaN,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,Archaeoglobus,Archaeoglobus fulgidus,Archaeoglobus fulgidus (strain ATCC 49558 / DS...,207.0,Wider GTDB 214
WP_148183473.1@GCF_000008665.1,NaN,NaN,WP_148183473.1@GCF_000008665.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,Archaeoglobus,Archaeoglobus fulgidus,Archaeoglobus fulgidus DSM 4304,195.0,DB prokaryotes (GTDB 214 subset)
MCS7130230.1@GCA_025058955.1,NaN,NaN,MCS7130230.1@GCA_025058955.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,263.0,DB prokaryotes (GTDB 214 subset)
MCS7131040.1@GCA_025058955.1,NaN,NaN,MCS7131040.1@GCA_025058955.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,89.0,DB prokaryotes (GTDB 214 subset)
JANXDV010000019.1_2,NaN,JANXDV010000019.1_2,NaN,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,NaN,264.0,Wider GTDB 214


In [86]:
halc8_hits['domain'].value_counts()

Archaea     106
Bacteria     31
Name: domain, dtype: int64

## Find NCBI accession for non DB Proka hits

In [87]:
def find_ncbi_assembly_assembly(contig_accession, email='rs1521@ic.ac.uk'):
    Entrez.email = email

    # Step 1: esearch - Find the UID for the contig
    search_handle = Entrez.esearch(db="nucleotide", term=contig_accession)
    search_record = Entrez.read(search_handle)
    search_handle.close()
    
    if not search_record["IdList"]:
        return None
    
    contig_uid = search_record["IdList"][0]

    # Step 2: elink - Find all linked UIDs in the assembly database
    link_handle = Entrez.elink(dbfrom="nucleotide", db="assembly", id=contig_uid)
    link_record = Entrez.read(link_handle)
    link_handle.close()
    
    assembly_links = link_record[0]["LinkSetDb"]
    if not assembly_links:
        return None

    assembly_uids = [link['Id'] for link in assembly_links[0]['Link']]

    # Step 3: esummary - Fetch summaries for all linked assembly UIDs
    summary_handle = Entrez.esummary(db="assembly", id=",".join(assembly_uids))
    summary_record = Entrez.read(summary_handle)
    summary_handle.close()

    # Step 4: Filter summaries to find the RefSeq (GCF_) record
    all_summaries = summary_record['DocumentSummarySet']['DocumentSummary']

    if len(all_summaries) > 0:
        for summary in all_summaries:
            return summary['AssemblyAccession']
    else:
        return None

In [88]:
protein_id_to_genome_accession = {}
contig_id_to_genome_accession = {}

protein_ids = halc8_hits[halc8_hits['db_proka_id'].isnull()]['gtdb_id'].unique()
for i, protein_id in enumerate(protein_ids):
    contig_id = re.match(r'^([^\.]+\.[0-9]+)_.+$', protein_id)[1]
    
    if contig_id in contig_id_to_genome_accession:
        protein_id_to_genome_accession[protein_id] = contig_id_to_genome_accession[contig_id]

    assembly_accession = find_ncbi_assembly_assembly(contig_id)

    if assembly_accession is not None:
        protein_id_to_genome_accession[protein_id] = assembly_accession
        contig_id_to_genome_accession[contig_id] = assembly_accession

    print(f'{i+1:,} of {len(protein_ids):,} | {contig_id} | {assembly_accession}')

    time.sleep(0.5)

print(len(protein_id_to_genome_accession))

1 of 56 | NC_000917.1 | GCF_000008665.1
2 of 56 | JANXDV010000019.1 | None
3 of 56 | CP100465.1 | GCF_024227995.1
4 of 56 | NZ_CP035119.1 | GCF_004087835.1
5 of 56 | NZ_CP035119.1 | GCF_004087835.1
6 of 56 | NZ_VSLZ01000004.1 | GCF_010747475.1
7 of 56 | NZ_BIXZ01000012.1 | GCF_005406125.1
8 of 56 | NZ_CAAHFB010000001.1 | GCF_900880625.1
9 of 56 | NZ_VRYN01000005.1 | GCF_008124605.1
10 of 56 | NZ_AOLJ01000018.1 | GCF_000336775.1
11 of 56 | NZ_FOXI01000014.1 | GCF_900115675.1
12 of 56 | CP002989.1 | GCF_000224475.1
13 of 56 | NZ_AP017570.1 | GCF_002355655.1
14 of 56 | NZ_JABUQZ010000001.1 | GCF_013342145.1
15 of 56 | NZ_RQWN01000003.1 | GCF_003977755.1
16 of 56 | NZ_AOIS01000013.1 | GCF_000337495.1
17 of 56 | NZ_AOIS01000017.1 | GCF_000337495.1
18 of 56 | NZ_JNCS01000004.1 | GCF_000731985.1
19 of 56 | NZ_CP058601.1 | GCF_013402815.2
20 of 56 | NZ_CP058601.1 | GCF_013402815.2
21 of 56 | NZ_AOII01000014.1 | GCF_000337615.1
22 of 56 | NZ_CP084472.1 | GCF_020405225.1
23 of 56 | AOIR01000038.

In [ ]:
halc8_gtdb = halc8_hits[
    halc8_hits['db_proka_id'].notnull() |
    halc8_hits['gtdb_id'].isin(sorted(protein_id_to_genome_accession.keys()))
].reset_index()

def set_id(row):
    if not pd.isnull(row['db_proka_id']):
        return row['db_proka_id']
    else:
        protein_id = row['gtdb_id']
        assembly_accession = protein_id_to_genome_accession[protein_id]
        return f'{protein_id}@{assembly_accession}'

halc8_gtdb['old_id'] = halc8_gtdb['id']
halc8_gtdb['id'] = halc8_gtdb.apply(
    set_id,
    axis=1,
)
halc8_gtdb['protein_id'] = halc8_gtdb['id'].apply(lambda v: v.split('@')[0])
halc8_gtdb['assembly_accession'] = halc8_gtdb['id'].apply(lambda v: v.split('@')[1])
halc8_gtdb['length'] = halc8_gtdb['length'].astype(int)

for accession in ['GCF_024227995.1', 'GCF_000224475.1', 'GCF_000337115.1', 'GCF_005954745.1', 'GCF_943914015.1']:
    new_accession = accession.replace('GCF_', 'GCA_')
    row = halc8_gtdb[halc8_gtdb['assembly_accession'] == accession].iloc[0]
    ix = row.name
    protein_id = row['protein_id']
    halc8_gtdb.loc[ix, 'assembly_accession'] = new_accession
    halc8_gtdb.loc[ix, 'id'] = f'{protein_id}@{new_accession}'

# Remove duplicated sequence
halc8_gtdb = halc8_gtdb[halc8_gtdb['assembly_accession'] != 'GCA_943913325.1'].copy()

# Remove a few duplicates due to slightly different ORF predictions between GTDB (Prodigal) and RefSeq
a = halc8_gtdb[halc8_gtdb['assembly_accession'].str.startswith('GCF_') & ~(halc8_gtdb['protein_id'].str.startswith('WP_'))]['assembly_accession'].unique()
b = halc8_gtdb[halc8_gtdb['assembly_accession'].isin(a) & halc8_gtdb['protein_id'].str.startswith('WP_')]['assembly_accession'].unique()
ids_to_remove = halc8_gtdb[halc8_gtdb['assembly_accession'].isin(b) & ~(halc8_gtdb['protein_id'].str.startswith('WP_'))]['id'].unique()
halc8_gtdb = halc8_gtdb[~(halc8_gtdb['id'].isin(ids_to_remove))].copy()

# Special treatment for Natrinema altunense:
# Leader sequence present in genome but incorrectly called in deposited genome. 
# We add it in later, so let's update its length
ix = halc8_gtdb[halc8_gtdb['id'] == 'WP_007110057.1@GCF_000731985.1'].iloc[0].name
halc8_gtdb.loc[ix, 'length'] = 283

halc8_gtdb = halc8_gtdb.drop_duplicates('id').drop(columns=['gtdb_id', 'db_proka_id'])[[
    'id', 'assembly_accession', 'protein_id', 
    'domain', 'gtdb_phylum', 'gtdb_class', 'gtdb_order', 'gtdb_family', 
    'gtdb_genus', 'gtdb_species', 'ncbi_organism_name', 'uniprot_id', 'length', 'old_id',
]].set_index('id')

In [98]:
gtdb_metadata_all = pd.concat(
    [
        pd.read_csv(data_folder / 'gtdb_r214.1' / 'ar53_metadata_r214.tsv', sep='\t'),
        pd.read_csv(data_folder / 'gtdb_r214.1' / 'bac120_metadata_r214.tsv', sep='\t'),
    ], 
    ignore_index=True,
)
gtdb_metadata_all['assembly_accession'] = gtdb_metadata_all['accession'].apply(lambda v: v[3:])
gtdb_metadata_all = gtdb_metadata_all.set_index('assembly_accession')

gtdb_metadata = gtdb_metadata_all.loc[halc8_gtdb['assembly_accession'].unique()].copy()
gtdb_metadata_all = None

gtdb_metadata['domain'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['domain'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_phylum'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_phylum'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_class'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_class'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_order'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_order'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_family'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_family'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_genus'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_genus'] for a in gtdb_metadata.index]
gtdb_metadata['gtdb_species'] = [halc8_gtdb[halc8_gtdb['assembly_accession'] == a].iloc[0]['gtdb_species'] for a in gtdb_metadata.index]

gtdb_metadata.to_csv(halc8_out / 'sequences' / 'HalC8_GTDB_r214_metadata.csv')

/opt/homebrew/Caskroom/miniforge/base/envs/amp/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3444: DtypeWarning: Columns (61,63,65,74,82,83) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [99]:
halc8_gtdb['ncbi_organism_name'] = halc8_gtdb['assembly_accession'].apply(lambda a: gtdb_metadata.loc[a, 'ncbi_organism_name'])

halc8_gtdb[[c for c in halc8_gtdb.columns if c != 'old_id']].to_csv(halc8_out / 'sequences' / 'HalC8_GTDB_r214_hits.csv')

halc8_gtdb.head()

,assembly_accession,protein_id,domain,gtdb_phylum,gtdb_class,gtdb_order,gtdb_family,gtdb_genus,gtdb_species,ncbi_organism_name,uniprot_id,length,old_id
id,,,,,,,,,,,,,
WP_148183473.1@GCF_000008665.1,GCF_000008665.1,WP_148183473.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,Archaeoglobus,Archaeoglobus fulgidus,Archaeoglobus fulgidus DSM 4304,NaN,195,WP_148183473.1@GCF_000008665.1
MCS7130230.1@GCA_025058955.1,GCA_025058955.1,MCS7130230.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,NaN,263,MCS7130230.1@GCA_025058955.1
MCS7131040.1@GCA_025058955.1,GCA_025058955.1,MCS7131040.1,Archaea,Halobacteriota,Archaeoglobi,Archaeoglobales,Archaeoglobaceae,WYZ-LMO2,WYZ-LMO2 sp025058955,Archaeoglobaceae archaeon,NaN,89,MCS7131040.1@GCA_025058955.1
WP_139025500.1@GCF_000376445.1,GCF_000376445.1,WP_139025500.1,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haladaptataceae,Haladaptatus,Haladaptatus paucihalophilus,Haladaptatus paucihalophilus DX253,A0A1M7C1A1,287,A0A1M7C1A1
WP_082837828.1@GCF_001625445.1,GCF_001625445.1,WP_082837828.1,Archaea,Halobacteriota,Halobacteria,Halobacteriales,Haladaptataceae,Haladaptatus,Haladaptatus sp001625445,Haladaptatus sp. R4,NaN,266,WP_082837828.1@GCF_001625445.1


### Export fasta sequences

In [ ]:
halc8_gtdb_records = []
for record in SeqIO.parse(halc8_out / 'sequences' / 'HalC8_proteins.fasta', 'fasta'):
    old_id = record.id.split('$')[0]
    df = halc8_gtdb[halc8_gtdb['old_id'] == old_id]
    if len(df) == 0:
        continue

    row = df.iloc[0]
    new_id = row.name
    gtdb_species = gtdb_metadata.loc[row['assembly_accession'], 'gtdb_species']

    record.id = new_id
    record.name = ''
    record.description = f'[{gtdb_species}]'

    if new_id == 'WP_007110057.1@GCF_000731985.1':
        # Special treatment for Natrinema altunense:
        # Add leader sequence present in genome but incorrectly called in deposited genome
        leader = 'MKEDNNTSEESGRINRRNVLKTVGAAGLFAAGSTG'
        record.seq = leader + record.seq
        record.description = f'[{gtdb_species}] [MODIFIED]'

    halc8_gtdb_records.append(record)

assert len(halc8_gtdb_records) == len(halc8_gtdb)

with (halc8_out / 'sequences' / 'HalC8_GTDB_r214_hits.fasta').open('w') as f_out:
    SeqIO.write(halc8_gtdb_records, f_out, 'fasta')

### Addendum: Fix protein ID of RefSeq sequences

GTDB protein IDs are prodigal based but we want to use the RefSeq IDs where possible.

In [61]:
halc8_hits = pd.read_csv(halc8_folder / 'sequences' / 'HalC8_GTDB_r214_hits__backup.csv', index_col='id')

def parse_tblout(path):
    return pd.read_csv(
        path, 
        sep='\s+', 
        comment='#', 
        header=None,
        usecols=[0, 2, 5],
        names=[
            'target', 'query', 'score',
        ],
    )

map_to_genome = parse_tblout(halc8_folder / 'sequences' / 'map_to_genome.tblout.txt')
map_to_genome['query_protein_id'] = map_to_genome['query'].apply(lambda v: v.split('@')[0])
map_to_genome['protein_id'] = map_to_genome['target'].apply(lambda t: t.split('@')[0])
map_to_genome['query_accession'] = map_to_genome['query'].apply(lambda v: v.split('@')[1])
map_to_genome['accession'] = map_to_genome['target'].apply(lambda t: t.split('@')[1].split('$')[0])
map_to_genome['new_id'] = map_to_genome['target'].apply(lambda t: t.split('$')[0])
map_to_genome['is_same_protein'] = map_to_genome['query_protein_id'] == map_to_genome['protein_id']
map_to_genome['is_same_genome'] = map_to_genome['query_accession'] == map_to_genome['accession']
map_to_genome = map_to_genome[map_to_genome['is_same_genome']].sort_values(
    ['query', 'is_same_protein', 'score'],
    ascending=[True, False, False],
).drop_duplicates('query').set_index('query')
map_to_genome.head()

,target,score,query_protein_id,protein_id,query_accession,accession,new_id,is_same_protein,is_same_genome
query,,,,,,,,,
AEN07551.1@GCA_000224475.1,AEN07551.1@GCA_000224475.1$Halolamina_sp000224475,647.5,AEN07551.1,AEN07551.1,GCA_000224475.1,GCA_000224475.1,AEN07551.1@GCA_000224475.1,True,True
ATMD01000005.1_21@GCA_001856825.1,EQB64365.1@GCA_001856825.1$GCA-001856825_sp001...,448.5,ATMD01000005.1_21,EQB64365.1,GCA_001856825.1,GCA_001856825.1,EQB64365.1@GCA_001856825.1,False,True
CALUAK010000087.1_2@GCA_943914015.1,CALUAK010000087.1_2@GCA_943914015.1$Corynebact...,254.5,CALUAK010000087.1_2,CALUAK010000087.1_2,GCA_943914015.1,GCA_943914015.1,CALUAK010000087.1_2@GCA_943914015.1,True,True
CP002989.1_262@GCA_000224475.1,AEN07551.1@GCA_000224475.1$Halolamina_sp000224475,647.5,CP002989.1_262,AEN07551.1,GCA_000224475.1,GCA_000224475.1,AEN07551.1@GCA_000224475.1,False,True
CP040678.1_2475@GCA_005954745.1,CP040678.1_2475@GCA_005954745.1$Halostella_pel...,640.6,CP040678.1_2475,CP040678.1_2475,GCA_005954745.1,GCA_005954745.1,CP040678.1_2475@GCA_005954745.1,True,True


In [68]:
ids = map_to_genome[map_to_genome.duplicated('new_id')]['new_id'].unique()
ids_to_fix = map_to_genome[
    map_to_genome['new_id'].isin(ids) &
    (~(map_to_genome['is_same_protein']))
].sort_values('new_id').index
for id_ in ids_to_fix:
    map_to_genome.loc[id_, 'new_id'] = id_

assert len(map_to_genome[map_to_genome.duplicated('new_id')]['new_id'].unique()) == 0

In [76]:
map_to_genome[map_to_genome['query_accession'] == 'GCF_900636325.1']

,target,score,query_protein_id,protein_id,query_accession,accession,new_id,is_same_protein,is_same_genome
query,,,,,,,,,


In [69]:
halc8_hits_ = halc8_hits.reset_index()
halc8_hits_['old_id'] = halc8_hits_['id'].copy()
halc8_hits_['protein_id'] = halc8_hits_.apply(
    lambda row: map_to_genome.loc[row['id'], 'protein_id'] if row['id'] in map_to_genome.index else row['protein_id'],
    axis=1
)
halc8_hits_['id'] = halc8_hits_['id'].apply(
    lambda id_: map_to_genome.loc[id_, 'new_id'] if id_ in map_to_genome.index else id_
)
halc8_hits_ = halc8_hits_.set_index('id')

assert len(set(halc8_hits_.index)) == len(halc8_hits_.index)

halc8_hits_[[c for c in halc8_hits_.columns if c != 'old_id']].to_csv(halc8_out / 'sequences' / 'HalC8_GTDB_r214_hits.csv')

In [70]:
halc8_gtdb_records = []
for record in SeqIO.parse(halc8_folder / 'sequences' / 'HalC8_GTDB_r214_hits__backup.fasta', 'fasta'):
    old_id = record.id.split('$')[0]
    df = halc8_hits_[halc8_hits_['old_id'] == old_id]
    if len(df) == 0:
        continue

    row = df.iloc[0]
    new_id = row.name
    record.id = new_id
    halc8_gtdb_records.append(record)

assert len(halc8_gtdb_records) == len(halc8_hits_)

with (halc8_out / 'sequences' / 'HalC8_GTDB_r214_hits.fasta').open('w') as f_out:
    SeqIO.write(halc8_gtdb_records, f_out, 'fasta')